## **Advanced Pandas**

### **In this lesson you:**
* Explore some more advanced features pandas provides including:
    - Renaming columns
    - Filtering the DataFrame
    - Grouping and aggregation Functions
    - Sorting
    - Imputing columns

Remember, to access `pandas` functionality, we must `import` the library first. We do not have to `pip install pandas` because we already have it installed.

In [ ]:
import pandas as pd

#### **Reading data** 

So far we have created a DataFrame by manually specifying the rows and columns. Often, we will have a datase stored as a CSV (comma-separated-value) file.

`pandas` provides a function called **read_csv(path)**, where we provide a path to where our CSV file is stored, and it returns a DataFrame of the contents at that path.

You'll be analyzing data from **Inside Airbnb** to better understand the San Fransisco rental market. Let's read in the dataset.

In [ ]:
file_path = "/Users/eryk/code/databricks-itp/resources/sf_airbnb_listings.csv"
df = pd.read_csv(file_path)

To look at the first few records of the datase, we can call **head()**. If you do not specify the number of rows, it defaults to 5 rows.

In [ ]:
# df.head()
df.head(3)

Coversaely, we can call **tail()** to look at the last few records.

In [ ]:
df.tail(3)

#### **Renaming Columns**

We can rename columns of our DataFrame using **rename()**. We pass into columns a dictionary containing the mappings from the old column names to the new ones to the `columns`parameter.

Let's rename the `neighbourhood` column above to be `neighbordhood` .

In [ ]:
df = df.rename(columns={"neighbourhood": "neighborhood"})
df.head(3)

####  **Filtering**

Often, you will want to select a subset of rows that meet a certain criteria, which can be accomplished by specifying: `df[bool_array]`, where `bool_array` si a `Series` of `True` and `False` values for each row.

The rows that evaluate to `True` are kept, while the ones that evaluate to `False` are not.

Let's filter for all teh rows `host_is_superhost` is `t`, meaning the airbnb owner is a superhost.

In [ ]:
filtered_df = df[df["host_is_superhost"] == "t"]
filtered_df.head(3)

Here, `df["host_is_superhost"] == "t"` is our boolean array. Let's take a look at the corresponding `True/False` row indices. 

In [ ]:
df["host_is_superhost"] == "t"

We can also search for all the records where the `host_is_superhost` is NOT `"t"`

In [ ]:
df["host_is_superhost"] != "t"

####  **Pandas Boolean Operators**

Often you will want to evaluate multiple criteria to filter out records. For exmaple, let's select all records where the host is a superhost and the airbnb has at least 150 reviews.

Instead of the normal Boolean operators we have seen previously, we have [bitwise Boolean operators](https://realpython.com/python-bitwise-operators/):

* `and` -> `&`
* `or` -> `|`
* `not` -> `~`

In [ ]:
filtered_df = df[(df["host_is_superhost"] == "t") & (df["number_of_reviews"] >= 150)]
filtered_df.head(3)

#### **Aggregate Funcions**

Aggregate functions are functions that take in a series of inputs and return a single output.

The most common ones that we use in pandas are ones that take in numerical `Series` and return a statistic of interest, such as the mean.

Let's take a look at the mean, min, and max of `number_of_reviews`:

In [ ]:
print(df["number_of_reviews"].mean())
print(df["number_of_reviews"].min())
print(df["number_of_reviews"].max())

Another useful method is [describe()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.describe.html) which provides a report of summary statistics on a given numerical `Series`:

In [ ]:
df["number_of_reviews"].describe()

We can also use this method on a DataFrame to see it applied to every numerical column:

In [ ]:
df[["number_of_reviews", "host_listings_count", "bedrooms"]].describe()

Many times, you won't care about the 6th value after the decimal. Let's round our results by calling [round()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.round.html#pandas.DataFrame.round).

In [ ]:
df[["number_of_reviews", "host_listings_count", "bedrooms"]].describe().round(2)

#### **Group By**

Sometimes we will want to see the results of an aggregate function per category in a non-numerical column.

For example, say we want to see the average number of bedrooms per neighborhood.

In order to do this we first use the [groupby([columns])](https://pandas.pydata.org/docs/reference/groupby.html) method and specify the category we want to group by. In this case, let's group by `neighborhood`.

In [ ]:
df.groupby(["neighborhood"]).mean(numeric_only=True).head(10)[["bedrooms"]]

Then apply teh aggregate function of interest. In this case, `mean()` to the column of interest, in this case `bedrooms`.

**Note:** Here we use `[["bedrooms"]]` to select for bedrooms because we could add other columns in addition to bedrooms.

In [ ]:
grouped_df = df.groupby(["neighborhood"])[["bedrooms"]].mean().head(10)
grouped_df

#### **Reset Index**

Typically, row indices are numbers, but they can also be named. In the example above, `neighborhood` is now the row index, rather than a column.

We can see that if we print out the columns.

In [ ]:
grouped_df.columns

To reset the index to be numbers, and move the current index to be a column, we use [reset_index()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.reset_index.html#pandas.DataFrame.reset_index) .

In [ ]:
reset_df = grouped_df.reset_index()
reset_df

####  **Sorting**

Pandas provide a [sort_values()](https://pandas.pydata.org/docs/search.html?q=sort_values#) method to sort the rows in a `DataFrame` or `Series`.

If called on a `DataFrame` you need to specify which column you are sorting by like this `df.sort_values([col])`

In [ ]:
df.sort_values(["bedrooms"]).head(3)

If applied to a `Series` there is only one column, so you don't need to specify.

In [ ]:
df["bedrooms"].sort_values()

By default `sort_values()` sorts in ascending order. You can specify the `ascending=False` parameter to change it to descending order.

In [ ]:
df["bedrooms"].sort_values(ascending=False)

####  **NaN**

You might have noticed that our `DataFrame` contains NaN values. These indicate a missing value.

We have a few ways we can handle missing values. Often having these values present causes problems for computational tasks.

First, we can check using the [isna()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.isna.html#pandas.DataFrame.isna) method, alias for [isnull()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.isnull.html#pandas.DataFrame.isnull), and the `sum()` method to count the number of NaN values present.

In [ ]:
nan_df = df[["security_deposit", "notes"]] # subset of columns with NaNs
nan_df

In [ ]:
nan_df.isna().sum()

#### **Dropping NaN**

One way you can handle NaN is to drop all rows that have NaN values. We can use the [dropna()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.dropna.html#pandas.DataFrame.dropna) method to do that.

In [ ]:
nan_df.dropna()

#### **Input columns**

However, we are throwing away a lot of information when we drop records - in the example above, we removed over 3000 rows.

Instead of dropping rows with missing values, we can impute the missing values using [fillna()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.fillna.html#pandas.DataFrame.fillna) and specifying a default value to use.

In [ ]:
nan_df.fillna("Missing")


Oftentimes, we want to impute different values to different columns. For example, with `numeric`values, we can impute with the mean/median/etc. For `categorical` features, imputing with the mode or a special category are common.

Let's instead that **`security_deposit`** is `$0.00` if it is missing. We can pass in a dictionary to **`fillna()`** that has column names as the key and the value to impute the column with as the value.

You can optionally specify `inplace=True` if you want to update the underlying DataFrame.

In [ ]:
nan_df.fillna({"security_deposit": "$0.00", "notes": "Missing"}, inplace=False)

####   **Write to CSV**

We can write a `pandas` DataFrame to a CSV file as shown below using the [to_csv()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_csv.html#pandas.DataFrame.to_csv) method.

In [ ]:
write_path = "./example.csv"
# dbutils.fs.rm(write_path) # remove if file exists
df.to_csv(write_path, index=False)

We can then read our csv file back in with **`read_csv`** .

In [ ]:
load_df = pd.read_csv(write_path)
load_df.head(3)